### Include Library

In [1]:
from datetime import datetime
import os
from multiprocessing import Pool
import math

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

### 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [3]:
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,  # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,  # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,  # or None
)

### 2. Load Data

In [5]:
print("Load caption file...")

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

# create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json(
    "data/low-quality_evaluation_5432-images_2025-04-10_15_29.json"
)
org_caption_dataset = org_caption_dataset[:4]

Load caption file...


### Run Multi Processors of steps 3 to 5
- step 3: generate atomics
- step 4: evaluate and get recall and precision
- step 5: calculate cap f1 score

In [10]:
def process_batch(
    start_idx,
    end_idx,
    org_caption_dataset,
    all_human_captions,
    folder_path,
    timestamp,
    chunk_id,
):
    subset = org_caption_dataset[start_idx:end_idx]
    LIMIT = len(subset)
    human_subset = all_human_captions[start_idx:end_idx]
    print(human_subset)

    # 3) generate atomics
    T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(subset, limit=LIMIT)
    # 3.1) save intermediate
    ResultsRepo.save_results_json(
        output_path=f"{folder_path}/intermediate_{timestamp}_chunk{chunk_id}.json",
        org_dataset=subset,
        T_atomics=T_atomics,
        g_atomics=g_atomics,
        parsed_T=parsed_T,
        T_org=human_subset,
        limit=LIMIT,
    )

    # 4) evaluate and get recall and precision
    eval_out = proc.evaluate_matching(human_subset, T_atomics, g_atomics)
    # 4.1) save evaluation results
    ResultsRepo.save_results_json(
        output_path=f"{folder_path}/eval_{timestamp}_chunk{chunk_id}.json",
        update_existing=f"{folder_path}/intermediate_{timestamp}_chunk{chunk_id}.json",
        metadata=eval_out,
        limit=LIMIT,
    )

    # 5) calculate cap f1 score
    cap_scores = proc.calculate_cap_f1(eval_out)

    # 5.1) save cap f1 score results
    ResultsRepo.save_results_json(
        output_path=f"{folder_path}/final_{timestamp}_chunk{chunk_id}.json",
        update_existing=f"{folder_path}/eval_{timestamp}_chunk{chunk_id}.json",
        evaluations=cap_scores,
        limit=LIMIT,
    )


def run_parallel_processing(
    org_caption_dataset, all_human_captions, folder_path, timestamp, num_workers=32
):
    total = len(org_caption_dataset)
    chunk_size = math.ceil(total / num_workers)

    with Pool(processes=num_workers) as pool:
        jobs = []
        for i in range(num_workers):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, total)
            jobs.append(
                pool.apply_async(
                    process_batch,
                    (
                        start_idx,
                        end_idx,
                        org_caption_dataset,
                        all_human_captions,
                        folder_path,
                        timestamp,
                        i,
                    ),
                )
            )

        for job in jobs:
            job.get()

In [11]:
all_human_captions = []
for item in org_caption_dataset[:4]:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)

run_parallel_processing(
    org_caption_dataset, all_human_captions, folder_path, timestamp, num_workers=4
)

[['A can of Coca Cola on a counter is shown for when one can use a nice, cold drink.', 'A black can of Coca Cola Zero calorie soda is on the counter near the coffee maker.', 'A kitchen counter the various items on top including a can of Coca-Cola, metal containers, and a teapot.', 'a black tin of Coca Cola placed on a black surface', 'Black counter with canisters, kettle and can of soda.']][['imagine how you would describe this image on the phone to a friend.', "the photographer's hand holding a round bottle with a black lid.", "canned food held by a man's fingers with the thumb visible", 'A person holding a food can with the rear of the label facing forward.', "A White man's hand holding a Vitamin Bottle."]][['candy with pink color to eat and enjoy and text appear on the package', 'a yellow color cheese is packed in a transparent nylon', "a sachet packaged of Farley' s Orange Slices", 'Tasty, orange, jelly candies are in this bag.', "Plastic bag of Farley's brand orange slice candies.

100%|██████████| 1/1 [00:12<00:00, 12.03s/it]


Saved JSON to: results/2025-08-09_23-02/parsed_caption_2025-08-09_23-02_chunk0.json


100%|██████████| 1/1 [00:12<00:00, 12.33s/it]


Saved JSON to: results/2025-08-09_23-02/parsed_caption_2025-08-09_23-02_chunk1.json


...

Saved JSON to: results/2025-08-09_23-02/final_2025-08-09_23-02_chunk3.json


100%|██████████| 1/1 [00:22<00:00, 22.91s/it]


Saved JSON to: results/2025-08-09_23-02/recall_precision_2025-08-09_23-02_chunk0.json


100%|██████████| 1/1 [00:00<00:00, 13315.25it/s]


Saved JSON to: results/2025-08-09_23-02/final_2025-08-09_23-02_chunk0.json


### 6. merge results from different processors

In [12]:
ResultsRepo.merge_json_chunks(
    output_file=f"{folder_path}/final_{timestamp}_merged.json",
    file_pattern=f"{folder_path}/final_{timestamp}_chunk*.json",
)

Merged 4 entries into results/2025-08-09_23-02/__final_2025-08-09_23-02_merged.json


### 7. Finally, save the results to a csv file

In [13]:
# 7) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}_merged.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

CSV file saved to: results/2025-08-09_23-02/__final_2025-08-09_23-02_merged.csv
